# CAML: main experiments

Load pretrained weights and evaluate the main numerical results:
- **Noise recovery:** CAML entries in Table 1 (HD and Grid).
- **Path integration:** Table 2 (HD, Grid, and Combined Place).

Run Setup first, then the experiment sections you need. This notebook runs
models directly; no saved measurements or plotting code are required.

In [1]:
from pathlib import Path
import csv
import json
import sys
import torch
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
if not (ROOT / "caml/model.py").is_file():
    raise FileNotFoundError("Open this notebook from the CAML repository directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from caml import evaluation as evaluate
from caml.io import sha256

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(4)
CHECKPOINTS = ROOT / "checkpoints"
RESULTS = ROOT / "results"
manifest = json.loads((CHECKPOINTS / "manifest.json").read_text())["files"]
for model in ("hd", "grid", "place_both"):
    for suffix in ("", "_bump_shifting"):
        name = f"checkpoints/caml_{model}{suffix}.pt"
        if name not in manifest or sha256(ROOT / name) != manifest[name]:
            raise ValueError(f"Checkpoint verification failed: {name}")
print("Device:", torch.cuda.get_device_name() if DEVICE == "cuda" else "CPU (full evaluation is slow)")

def energy(name):
    return CHECKPOINTS / f"caml_{name}.pt"

def shifting(name):
    return CHECKPOINTS / f"caml_{name}_bump_shifting.pt"

def save_csv(name, records):
    RESULTS.mkdir(exist_ok=True)
    with (RESULTS / name).open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)

def show(columns, rows):
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    lines += ["| " + " | ".join(map(str, row)) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

tracking_results = {}

def show_tracking():
    records = [r for population in ("hd", "grid", "place") for r in tracking_results.get(population, [])]
    save_csv("table2_tracking.csv", records)
    groups = {}
    for record in records:
        key = (record["population"], record["velocity_limit"])
        groups.setdefault(key, {})[record["method"]] = record
    rows = []
    for (population, limit), methods in groups.items():
        scale, unit = (1000, "mm") if population == "place" else (1, "degree")
        values = [f"{methods[m]['mean']*scale:.3f} ± {methods[m]['sd']*scale:.3f}" for m in ("learned", "fixed")]
        rows.append([population.upper() if population == "hd" else population.capitalize(), limit, unit, *values])
    show(["Population", "Velocity limit (rad/s)", "Unit", "Learned", "Fixed"], rows)

Device: NVIDIA RTX A4500


## Noise recovery — Table 1

Gaussian noise SD is 0.1 or 0.5 times the reference population RMS.
Recovery lasts **25 s for HD** and **200 s for Grid**, with RK4 and dt = 0.05 s.
Mean ± SD is computed across **10,000 HD states** or **2,500 Grid states**
per noise level, using each state's normalized population RMSE.

In [2]:
recovery_results = evaluate.recovery(energy("hd"), device=DEVICE)
recovery_results += evaluate.recovery(energy("grid"), device=DEVICE)
save_csv("table1_caml.csv", recovery_results)
show(["Population", "Noise SD / reference RMS", "Normalized population RMSE", "States"],
     [[r["population"].upper() if r["population"] == "hd" else "Grid", r["noise_sd"],
       f"{r['mean']:.3f} ± {r['sd']:.3f}", r["n"]] for r in recovery_results])

HD noise=0.1: 2000/10000 states
HD noise=0.1: 4000/10000 states
HD noise=0.1: 6000/10000 states
HD noise=0.1: 8000/10000 states
HD noise=0.1: 10000/10000 states
HD noise=0.5: 2000/10000 states
HD noise=0.5: 4000/10000 states
HD noise=0.5: 6000/10000 states
HD noise=0.5: 8000/10000 states
HD noise=0.5: 10000/10000 states
GRID noise=0.1: 500/2500 states
GRID noise=0.1: 1000/2500 states
GRID noise=0.1: 1500/2500 states
GRID noise=0.1: 2000/2500 states
GRID noise=0.1: 2500/2500 states
GRID noise=0.5: 500/2500 states
GRID noise=0.5: 1000/2500 states
GRID noise=0.5: 1500/2500 states
GRID noise=0.5: 2000/2500 states
GRID noise=0.5: 2500/2500 states


| Population | Noise SD / reference RMS | Normalized population RMSE | States |
| --- | --- | --- | --- |
| HD | 0.1 | 0.014 ± 0.009 | 10000 |
| HD | 0.5 | 0.051 ± 0.038 | 10000 |
| Grid | 0.1 | 0.078 ± 0.032 | 2500 |
| Grid | 0.5 | 0.079 ± 0.032 | 2500 |

## HD/Grid path integration — Table 2

Evaluate **100 paired random walks** at velocity limits 0.5 and 1.0 rad/s.
HD trajectories last 20 s; Grid trajectories last 40 s. The two Grid velocity
axes use independent random increments drawn from the same distribution.
Dynamics and velocity inputs update every **0.05 s**.

Both shifting methods receive the same inputs. Tracking errors are averaged
over time after the first 10% of each HD/Grid trajectory; Grid phase axes are
averaged within each trajectory before computing the mean and SD across trajectories.

In [3]:
tracking_results["hd"] = evaluate.path_integration(energy("hd"), shifting("hd"), device=DEVICE)
tracking_results["grid"] = evaluate.path_integration(energy("grid"), shifting("grid"), device=DEVICE)
show_tracking()

HD tracking: 40/400 steps
HD tracking: 80/400 steps
HD tracking: 120/400 steps
HD tracking: 160/400 steps
HD tracking: 200/400 steps
HD tracking: 240/400 steps
HD tracking: 280/400 steps
HD tracking: 320/400 steps
HD tracking: 360/400 steps
HD tracking: 400/400 steps
GRID tracking: 80/800 steps
GRID tracking: 160/800 steps
GRID tracking: 240/800 steps
GRID tracking: 320/800 steps
GRID tracking: 400/800 steps
GRID tracking: 480/800 steps
GRID tracking: 560/800 steps
GRID tracking: 640/800 steps
GRID tracking: 720/800 steps
GRID tracking: 800/800 steps


| Population | Velocity limit (rad/s) | Unit | Learned | Fixed |
| --- | --- | --- | --- | --- |
| HD | 0.5 | degree | 0.752 ± 0.375 | 0.654 ± 0.342 |
| HD | 1.0 | degree | 1.425 ± 0.653 | 1.557 ± 0.825 |
| Grid | 0.5 | degree | 0.093 ± 0.037 | 0.715 ± 0.287 |
| Grid | 1.0 | degree | 0.157 ± 0.050 | 1.968 ± 0.731 |

## Combined Place path integration — Table 2

Evaluate **100 paired trajectories of 20 s**, with dt = 0.05 s, restoring
gain 2, and the bounded-map position decoder. Report the mean and SD of
per-trajectory mean position errors. Only the Combined Place model is needed.

In [4]:
tracking_results["place"] = evaluate.place_tracking(
    energy("place_both"), shifting("place_both"), device=DEVICE)
show_tracking()

  PlacePopulation: built FD operators D_x, D_y (256×256, dx=0.05)
Place tracking: 1/100 trajectories
Place tracking: 2/100 trajectories
Place tracking: 3/100 trajectories
Place tracking: 4/100 trajectories
Place tracking: 5/100 trajectories
Place tracking: 6/100 trajectories
Place tracking: 7/100 trajectories
Place tracking: 8/100 trajectories
Place tracking: 9/100 trajectories
Place tracking: 10/100 trajectories
Place tracking: 11/100 trajectories
Place tracking: 12/100 trajectories
Place tracking: 13/100 trajectories
Place tracking: 14/100 trajectories
Place tracking: 15/100 trajectories
Place tracking: 16/100 trajectories
Place tracking: 17/100 trajectories
Place tracking: 18/100 trajectories
Place tracking: 19/100 trajectories
Place tracking: 20/100 trajectories
Place tracking: 21/100 trajectories
Place tracking: 22/100 trajectories
Place tracking: 23/100 trajectories
Place tracking: 24/100 trajectories
Place tracking: 25/100 trajectories
Place tracking: 26/100 trajectories
Place t

| Population | Velocity limit (rad/s) | Unit | Learned | Fixed |
| --- | --- | --- | --- | --- |
| HD | 0.5 | degree | 0.752 ± 0.375 | 0.654 ± 0.342 |
| HD | 1.0 | degree | 1.425 ± 0.653 | 1.557 ± 0.825 |
| Grid | 0.5 | degree | 0.093 ± 0.037 | 0.715 ± 0.287 |
| Grid | 1.0 | degree | 0.157 ± 0.050 | 1.968 ± 0.731 |
| Place | random_walk | mm | 3.918 ± 2.312 | 67.115 ± 70.021 |